# Démonstration finale — Système multi-agents Hypercare (AQUA / SpotOn)

**Projet TAL - M2 ISD • Étape 12 • Notebook de soutenance**

> **Données et code de référence :** `src/graph.py`, `src/nodes/`, `src/trace.py`, `src/workflows/kb_repair.py`, `tests/scenarios.json`. Les tickets S1–S7 sont des **scénarios de démonstration**, distincts des 100 tickets réservés dans `EVAL_TICKETS`.

**Règles de sécurité :** aucune modification de prix/réservation ; pas d'explication de prix sans preuve ; aucune publication automatique d'une hypothèse métier. Les opérations d'écriture dans la KB sont contrôlées par des cellules distinctes.

## 0. Prérequis et architecture

Exécution prévue **dans un notebook Python Snowflake**, ouvert depuis le dossier `notebooks/` du workspace du projet. L'environnement doit pouvoir importer le package `src`, accéder à `PROJECT_DB.PUBLIC`, et appeler le modèle configuré par `src/llm.py`.

**Graphe réellement présent dans le ZIP :** `intake → classify → doc_search → [data_investigation si nécessaire] → decide → answer/clarify/escalate`, avec les raccourcis configurés dans `src/graph.py`. Le nœud `resolve_refs` du schéma discuté précédemment **n'est pas présent dans cette version du ZIP** ; ne pas le présenter comme exécuté tant qu'il n'est pas intégré au code.

- **Agent documentaire** : recherche dans la KB RAG + validation de pertinence par LLM.
- **Agent données** : devis, réservation, lignes BAF09, deuxième extraction, référentiels.
- **Orchestrateur `decide`** : règles explicites : *répondre / préciser / escalader*.
- **Agents rédacteurs** : produisent la réponse ou le dossier à partir des éléments vérifiables.
- **Trace** : journal des outils et sources consultés, sans raisonnement privé du modèle.

In [ ]:
!pip install langgraph langchain-openai

In [ ]:
# À exécuter une seule fois si les paquets manquent dans l'environnement Snowflake.
# %pip install langgraph langchain-openai rank_bm25
# Après installation, redémarrer le kernel avant la cellule suivante.

import os
import sys
import json
import importlib
from pathlib import Path
from IPython.display import Image, display, Markdown
import pandas as pd

# Dans le workspace habituel : notebooks/ à côté de src/ et tests/.
PROJECT_ROOT = Path("..").resolve()
assert (PROJECT_ROOT / "src" / "graph.py").exists(), (
    "src/graph.py introuvable. Ouvrir ce notebook depuis notebooks/ "
    "ou adapter PROJECT_ROOT."
)
assert (PROJECT_ROOT / "tests" / "scenarios.json").exists(), (
    "tests/scenarios.json introuvable : vérifier le workspace."
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from snowflake.snowpark.context import get_active_session
from src.workflows.kb_repair import rollback_proposal
from src.workflows.kb_repair import publish_proposal
from src.llm import make_llm, MODEL
from src.graph import build_graph
from src.trace import show_trace, trace_rows
from src.workflows.kb_repair import (
    create_kb_proposal, approve_proposal, reject_proposal,
    publish_proposal, rollback_proposal
)

session = get_active_session()
with (PROJECT_ROOT / "tests" / "scenarios.json").open(encoding="utf-8") as f:
    SCENARIOS = json.load(f)["scenarios"]
BY_ID = {s["id"]: s for s in SCENARIOS}

print("Scénarios chargés :", len(SCENARIOS))
print("LLM :", MODEL)
print("Base Snowflake :", session.sql("SELECT CURRENT_DATABASE() AS DB").collect()[0]["DB"])

### Vérification non destructive des sources

On confirme l'existence des tables utilisées **sans les modifier**. Les tables de la partie D peuvent être absentes si le workflow 9 n'a pas été synchronisé avec le ZIP : cela ne bloque pas les scénarios S1–S4.

In [ ]:
TABLES = [
    "BASE_CONNAISSANCE_RAG", "KB_QA", "QUOTES", "BOOKINGS",
    "PRICELINES_BAF09", "PRICELINES_BAF09_UPDATE",
    "REF_CAR_CHARGES", "REF_GOL_SERVICES",
    "KB_PROPOSALS", "KB_VERSIONS",
    "SIMILAR_TICKET_ALERTS", "WORKFLOW_RUNS",
]
rows = session.sql("""
    SELECT TABLE_NAME
    FROM PROJECT_DB.INFORMATION_SCHEMA.TABLES
    WHERE TABLE_SCHEMA = 'PUBLIC'
""").collect()
existing = {r["TABLE_NAME"].upper() for r in rows}
checks = pd.DataFrame([{"Table": n, "Disponible": n in existing} for n in TABLES])
display(checks)
REQUIRED = ["BASE_CONNAISSANCE_RAG", "KB_QA", "QUOTES", "BOOKINGS",
            "PRICELINES_BAF09", "PRICELINES_BAF09_UPDATE"]
missing_required = [n for n in REQUIRED if n not in existing]
if missing_required:
    print("⚠️ Tables à vérifier avant la démonstration réelle :", missing_required)
else:
    print("✓ Tables principales disponibles")

### Initialiser **une seule fois** le graphe réel

**Important :** `build_graph(llm=llm, session=session)` active tous les nœuds réels présents dans ce ZIP. On n'utilise **pas** `fixtures`, réservées aux anciens tests factices. Après modification d'un module `src/`, **redémarrer le kernel** pour éviter d'exécuter une ancienne version.

In [ ]:
llm = make_llm()
graph = build_graph(llm=llm, session=session)
RESULTS = {}  # résultats conservés en mémoire pour la démonstration
print("✓ Graphe réel initialisé")

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))


### Afficher une trace complète

Le tableau ci-dessous reprend les entrées **réellement produites** par `src/trace.py` : nœud, agent, outils, éléments consultés, décision et résultat. Les champs documents et données montrent les sources fournies au système ; ils ne doivent pas être interprétés comme preuve que chaque source est fiable.

In [ ]:
def _as_text(v, limit=1100):
    value = str(v) if v is not None else ""
    return value if len(value) <= limit else value[:limit] + " … [tronqué pour affichage]"


def show_evidence(result):
    docs = result.get("docs") or []
    print(f"Documents candidats : {len(docs)} | source fiable : {result.get('docs_reliable', False)}")
    if docs:
        display(pd.DataFrame([
            {"Source": d.get("source") or d.get("source_id"),
             "Score": d.get("score"),
             "Extrait": _as_text(d.get("extrait", ""), 350)}
            for d in docs
        ]))
    data = result.get("data") or {}
    if data:
        print("Investigation opérationnelle :")
        print("  Nombre de lignes candidates :", data.get("nb_candidates"))
        print("  Preuve relevée :", data.get("evidence"))
        print("  Constat :", data.get("finding") or "Aucun constat suffisant")
        if data.get("quote"):
            print("  Devis consulté :", data["quote"].get("quote_id"))
        if data.get("booking"):
            print("  Réservation consultée :", data["booking"].get("booking_id"))
        change_info = (data.get("baf09_update") or {}).get("changes")
        if change_info:
            print("  Évolution BAF09 → BAF09_UPDATE :", change_info)
        if data.get("lines"):
            columns = ["detail_uid", "line_sequence", "charge_origin", "rate_structure",
                       "rate_20ST", "rate_40ST", "rate_40HC", "applicable", "fixed"]
            display(pd.DataFrame(data["lines"]).reindex(columns=columns))


def run_demo(scenario_id: str, show_details: bool = True):
    scenario = BY_ID[scenario_id]
    ticket = scenario["ticket"]
    print("=" * 84)
    print(f"{scenario_id} — {scenario.get('enonce', '')}")
    print("=" * 84)
    print("TICKET :", ticket.get("ticket_id"))
    print("QUESTION :", ticket.get("description"))
    print("ROUTE :", ticket.get("pol") or "non renseigné", "→", ticket.get("pod") or "non renseigné")
    print("DEVIS / RÉSERVATION :", ticket.get("quote_id") or "—", "/", ticket.get("booking_id") or "—")

    # Aucun champ 'fixtures' : uniquement les agents réels.
    result = graph.invoke({"ticket": ticket, "trace": []})
    RESULTS[scenario_id] = result
    print("\nDÉCISION :", result.get("decision"))
    print("RÈGLE :", result.get("decision_rule"))
    print("MOTIF :", result.get("decision_reason"))
    print("ÉQUIPE :", result.get("team"))
    print("LACUNE DOCUMENTAIRE :", result.get("kb_gap", False))
    print("\nRÉPONSE / MESSAGE / DOSSIER :\n", result.get("answer", "(aucun texte)"))

    if show_details:
        print("\n--- DOCUMENTS ET DONNÉES CONSULTÉS ---")
        show_evidence(result)
    print("\n--- TRACE OBSERVABLE (tous les agents) ---")
    trace = result.get("trace") or []
    if trace:
        display(pd.DataFrame(trace_rows(ticket.get("ticket_id", "?"), trace)))
        # Une seconde représentation verbale fidèle aux mêmes entrées.
        show_trace(trace)
    else:
        print("Aucune trace : vérifier le graphe.")

    expected = scenario.get("expected", {})
    checks = {
        "décision": (result.get("decision"), expected.get("decision")),
        "règle": (result.get("decision_rule"), expected.get("rule")),
        "kb_gap": (result.get("kb_gap", False), expected.get("kb_gap")),
    }
    print("\n--- CONFORMITÉ À L'ATTENDU DU SCÉNARIO ---")
    display(pd.DataFrame([{
        "Champ": k, "Observé": actual, "Attendu": wanted, "Conforme": actual == wanted
    } for k, (actual, wanted) in checks.items()]))
    return result

---
## 1. S1 — Question simple résolue à partir d'une source documentaire

**Situation :** l'utilisateur n'arrive pas à ajouter un voyage/service (KILIMA) à une pricelist. Le système doit rechercher dans la documentation, vérifier qu'un passage répond réellement à la question, puis fournir une réponse sourcée.

In [ ]:
r1 = run_demo("S1_question_documentaire")

---
## 2. S2 — Cas de prix nécessitant plusieurs données

**Situation :** le client observe une surcharge **BAF09** sur SpotOn alors qu'il pensait la charge incluse dans le fret.

**Ce qu'on cherche à démontrer :** consultation de plusieurs sources, notamment la documentation, les lignes `PRICELINES_BAF09`, l'extraction `PRICELINES_BAF09_UPDATE`, un devis et une réservation lorsqu'ils sont disponibles. La règle `decide` exige une **preuve issue des données** pour répondre à un cas de prix.

**Ce que la trace doit permettre de vérifier :** quels outils ont été appelés, quelles lignes et références ont été lues, quel constat a été produit, et pourquoi `repondre` (ou, en cas d'échec, `escalader`) a été choisi.

In [ ]:
r2 = run_demo("S2_prix_plusieurs_donnees")

In [ ]:
# Vue métier complémentaire : croisement des données réellement présentes en sortie de l'agent.
data2 = r2.get("data") or {}
summary2 = [
    {"Élément": "Lignes BAF09 candidates", "Valeur": data2.get("nb_candidates")},
    {"Élément": "Preuve explicative détectée", "Valeur": data2.get("evidence")},
    {"Élément": "Devis", "Valeur": (data2.get("quote") or {}).get("quote_id")},
    {"Élément": "Réservation", "Valeur": (data2.get("booking") or {}).get("booking_id")},
    {"Élément": "Nombre de lignes dans BAF09_UPDATE", "Valeur": (data2.get("baf09_update") or {}).get("nb_candidates")},
    {"Élément": "Comparaison UPDATE", "Valeur": (data2.get("baf09_update") or {}).get("changes") or "aucun changement relevé"},
]
display(pd.DataFrame(summary2))

---
## 3. S3 — Cas incertain correctement transmis à un expert

**Situation :** un écart entre le prix affiché dans SpotOn et le prix attendu dans AQUA n'est **pas expliqué** par les données BAF09 consultées.

**Attendu :** `decision = escalader`, `decision_rule = pas_de_preuve`. On affiche l'équipe destinataire, les éléments examinés et le dossier rédigé. C'est un résultat **positif pour la sûreté** du système : il évite une explication de prix inventée.

In [ ]:
r3 = run_demo("S3_cas_incertain")

---
## 4. S4 — Lacune documentaire et proposition de correction

**Situation :** le ticket demande si une suspension automatique hebdomadaire des offres SpotOn est documentée. Il s'agit d'une **question de test**, et **non d'une affirmation qu'une telle règle existe**.

**Attendu :** absence de réponse fiable → `escalader` + `kb_gap=True` → création d'une proposition de documentation en statut `pending` → **examen et décision d'un humain**.

> Le champ `kb_gap` n'est défini que pour le cas « pas de source fiable » dans `src/nodes/decide.py`. La détection automatique des contradictions et documents périmés est hors du périmètre de ce prototype.

In [ ]:
r4 = run_demo("S4_lacune_documentaire")
assert r4.get("kb_gap") is True, (
    "La démonstration KB requiert kb_gap=True. "
    "Vérifier la recherche et le routage avant de créer une proposition."
)
print("✓ Lacune signalée par le graphe. La réparation est un workflow séparé.")

### 4.1 Génération d'une proposition pour un humain

Cette cellule peut **écrire** dans `KB_PROPOSALS`. Elle recherche d'abord une proposition `pending` associée au ticket de démonstration afin d'éviter d'en créer plusieurs à chaque relance. La génération ne valide **aucune** règle métier.

**Attention :** les propositions du type « Ajouter une fiche KB précisant si la règle existe » sont des **demandes de documentation** ; il ne faut pas les publier comme des réponses factuelles aux utilisateurs.

In [ ]:
PROPOSAL_ID = None
S4_TICKET_ID = BY_ID["S4_lacune_documentaire"]["ticket"]["ticket_id"]

if "KB_PROPOSALS" not in existing:
    print("⚠️ KB_PROPOSALS absente. Exécuter d'abord la création des tables dans 10_kb_repair.ipynb.")
else:
    pending = session.sql("""
        SELECT PROPOSAL_ID
        FROM PROJECT_DB.PUBLIC.KB_PROPOSALS
        WHERE TICKET_ID = ? AND STATUS = 'pending'
        ORDER BY CREATED_AT DESC
        LIMIT 1
    """, params=[S4_TICKET_ID]).collect()
    if pending:
        PROPOSAL_ID = pending[0]["PROPOSAL_ID"]
        print("Proposition pending existante réutilisée :", PROPOSAL_ID)
    else:
        proposal = create_kb_proposal(session=session, llm=llm, state=r4)
        PROPOSAL_ID = proposal["proposal_id"] if proposal else None
        print("Nouvelle proposition créée :", PROPOSAL_ID)

    if PROPOSAL_ID:
        session.sql("""
            SELECT PROPOSAL_ID, TICKET_ID, QUESTION, PROBLEM_TYPE,
                   PROBLEM_DESCRIPTION, SUGGESTED_CONTENT,
                   SOURCES, STATUS, CREATED_AT
            FROM PROJECT_DB.PUBLIC.KB_PROPOSALS
            WHERE PROPOSAL_ID = ?
        """, params=[PROPOSAL_ID]).show()

In [ ]:
print("=== PROPOSITION DE CORRECTION ===")

display(Markdown(
    proposal["suggested_content"]
))

### 4.2 Point d'arrêt : **validation humaine explicite**

Dans un vrai processus métier, approuver **une fiche de travail** (par exemple « ce point doit être documenté ») ne suffit **pas** à valider son contenu comme une connaissance destinée aux utilisateurs. La publication exige une réponse éditoriale vérifiée par l'expert.

Pour démontrer le changement de statut, saisir volontairement un nom de relecteur et passer `VALIDER_MANUELLEMENT` à `True`, puis exécuter cette cellule une fois.

In [ ]:
# ÉTAPE HUMAINE : NE PAS ACTIVER PAR DÉFAUT.
VALIDER_MANUELLEMENT = True
RELECTEUR = "Javier Peña"  # Exemple : prénom / identifiant du relecteur présent
COMMENTAIRE_VALIDATION = (
    "Proposition de travail examinée : demander confirmation de la règle à l'expert métier. "
    "Ce statut n'autorise pas à publier une réponse factuelle non vérifiée."
)

if not VALIDER_MANUELLEMENT:
    print("⏸️ Pause : aucune validation effectuée. Expliquer le contrôle humain à l'oral.")
elif not PROPOSAL_ID:
    print("Aucune proposition disponible.")
elif not RELECTEUR.strip():
    raise ValueError("Renseigner RELECTEUR avant toute validation.")
else:
    approve_proposal(
        session=session,
        proposal_id=PROPOSAL_ID,
        reviewer=RELECTEUR.strip(),
        comment=COMMENTAIRE_VALIDATION,
    )
    print("Décision humaine enregistrée pour", PROPOSAL_ID)

if PROPOSAL_ID:
    session.sql("""
        SELECT PROPOSAL_ID, STATUS, REVIEWED_BY, REVIEW_COMMENT, REVIEWED_AT
        FROM PROJECT_DB.PUBLIC.KB_PROPOSALS
        WHERE PROPOSAL_ID = ?
    """, params=[PROPOSAL_ID]).show()

### 4.3 Variante : rejet humain (facultatif)

Cette cellule illustre un autre résultat possible du contrôle humain. **Ne jamais utiliser le même identifiant déjà approuvé** : seule une proposition encore `pending` peut être rejetée. À utiliser uniquement avec un identifiant distinct que l'on souhaite volontairement rejeter.

In [ ]:
proposal_rejet = create_kb_proposal(session=session, llm=llm, state=r4)
PROPOSAL_ID_REJET = proposal_rejet["proposal_id"]
print(PROPOSAL_ID_REJET)

In [ ]:
REJETER_MANUELLEMENT = True
PROPOSAL_ID_A_REJETER = PROPOSAL_ID_REJET # Autre proposition en attente, identifiée explicitement

if REJETER_MANUELLEMENT:
    if not PROPOSAL_ID_A_REJETER or not RELECTEUR.strip():
        raise ValueError("Indiquer un identifiant distinct et le relecteur.")
    if PROPOSAL_ID_A_REJETER == PROPOSAL_ID:
        raise ValueError("Ne pas rejeter la proposition approuvée lors de la démo.")
    reject_proposal(
        session=session,
        proposal_id=PROPOSAL_ID_A_REJETER,
        reviewer=RELECTEUR,
        comment="Proposition insuffisamment fondée : rejet après revue humaine.",
    )
    print("Proposition rejetée :", PROPOSAL_ID_A_REJETER)
else:
    print("Rejet manuel non activé : démonstration non destructive.")

In [ ]:
session.sql("""
    SELECT PROPOSAL_ID, TICKET_ID, STATUS, CREATED_AT
    FROM PROJECT_DB.PUBLIC.KB_PROPOSALS
    WHERE TICKET_ID = 'TCK-0004'
    ORDER BY CREATED_AT DESC
""").show()

In [ ]:
old_proposal_id = "--"

session.sql("""
    DELETE FROM PROJECT_DB.PUBLIC.KB_PROPOSALS
    WHERE PROPOSAL_ID = ?
      AND STATUS IN ('pending', 'approved', 'rejected')
""", params=[old_proposal_id]).collect()

---
## 5. Extension E — Versioning, publication et rollback (démonstration encadrée)

L'étape 10 inclut `publish_proposal()` et `rollback_proposal()` et la table `KB_VERSIONS`. 

- **Création** : `OLD_CONTENT = NULL`, `NEW_CONTENT = texte inséré`, `ACTION = insert`.
- **Annulation** : l'entrée créée est supprimée du RAG et la proposition passe à `rolled_back` ; la ligne d'historique demeure.
- **Limite actuelle importante** : le code publie `SUGGESTED_CONTENT` directement après approbation. Pour une vraie exploitation, exiger un **contenu final vérifié et distinct** des instructions « Ajouter une fiche… ». Il faut aussi préserver les entrées validées lors de toute reconstruction de l'index RAG.

Nous **ne réutilisons pas** automatiquement la proposition S4 pour une publication : personne n'a fourni de réponse métier attestée sur la suspension dominicale.

In [ ]:
from src.workflows.kb_repair import publish_proposal

publication = publish_proposal(
    session=session,
    proposal_id=PROPOSAL_ID
)

print("=== PUBLICATION ===")
print("Proposition :", publication["proposal_id"])
print("Source créée :", publication["source_id"])
print("Version :", publication["version_id"])
print("Statut :", publication["status"])

# Vérifier la fiche créée
session.sql("""
    SELECT FILE_NAME, SOURCE_TYPE, CONTENT
    FROM PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG
    WHERE FILE_NAME = ?
""", params=[publication["source_id"]]).show()

In [ ]:
publication_id = publication["proposal_id"]

rows = session.sql("""
    SELECT PROPOSAL_ID, STATUS
    FROM PROJECT_DB.PUBLIC.KB_PROPOSALS
    WHERE PROPOSAL_ID = ?
""", params=[publication_id]).collect()

for row in rows:
    print("ID :", row["PROPOSAL_ID"])
    print("Statut réel :", repr(row["STATUS"]))

In [ ]:
publication_id = publication["proposal_id"]
rollback = rollback_proposal(
    session=session,
    proposal_id=publication_id
)

print("=== ROLLBACK ===")
print("Proposition :", rollback["proposal_id"])
print("Source supprimée :", rollback["source_id"])
print("Statut :", rollback["status"])

# Vérifier que la fiche n'existe plus
session.sql("""
    SELECT COUNT(*) AS NB_FICHES
    FROM PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG
    WHERE FILE_NAME = ?
""", params=[rollback["source_id"]]).show()

# Vérifier que l'historique est conservé
session.sql("""
    SELECT VERSION_ID, ACTION, OLD_CONTENT, NEW_CONTENT
    FROM PROJECT_DB.PUBLIC.KB_VERSIONS
    WHERE PROPOSAL_ID = ?
""", params=[PROPOSAL_ID]).show()

In [ ]:
if "KB_VERSIONS" in existing and "KB_PROPOSALS" in existing:
    versions = session.sql("""
        SELECT v.VERSION_ID, v.PROPOSAL_ID, v.SOURCE_ID,
               v.SOURCE_TYPE, v.ACTION, v.CREATED_AT,
               p.STATUS AS PROPOSAL_STATUS,
               LEFT(v.NEW_CONTENT, 160) AS NEW_CONTENT_EXCERPT
        FROM PROJECT_DB.PUBLIC.KB_VERSIONS v
        LEFT JOIN PROJECT_DB.PUBLIC.KB_PROPOSALS p
          ON v.PROPOSAL_ID = p.PROPOSAL_ID
        ORDER BY v.CREATED_AT DESC
        LIMIT 10
    """).to_pandas()
    display(versions)
    if versions.empty:
        print("Aucune version enregistrée : vérifier le test dans 10_kb_repair.ipynb.")
    else:
        print("Historique consulté sans modification de la base.")
else:
    print("Tables KB absentes : cette extension doit être initialisée via 10_kb_repair.ipynb.")

---
## 6. Extension D — Détection automatique des tickets similaires

L'étape 9 a été expérimentée dans Snowflake avec `TASK_SIMILAR_TICKETS`, `SIMILAR_TICKET_ALERTS` et `WORKFLOW_RUNS`. **Le fichier `src/workflows/similar_tickets.py` et son notebook 09 ne sont pas inclus dans le ZIP de référence** : on montre ici les **résultats persistés dans Snowflake**

In [ ]:
# Consultation NON destructive des alertes déjà générées.
if "SIMILAR_TICKET_ALERTS" in existing:
    alerts_df = session.sql("""
        SELECT ALERT_ID, REFERENCE_TICKET_ID, TICKET_DATE,
               TICKET_IDS, TOTAL_TICKETS, MESSAGE, STATUS, CREATED_AT
        FROM PROJECT_DB.PUBLIC.SIMILAR_TICKET_ALERTS
        ORDER BY CREATED_AT DESC
        LIMIT 10
    """).to_pandas()
    display(alerts_df)
    if alerts_df.empty:
        print("Pas d'alerte enregistrée.")
else:
    print("SIMILAR_TICKET_ALERTS absente : récupérer les résultats depuis le notebook 09.")

if "WORKFLOW_RUNS" in existing:
    runs_df = session.sql("""
        SELECT RUN_AT, WORKFLOW_NAME, STATUS,
               TICKETS_ANALYZED, ALERTS_CREATED, DETAILS
        FROM PROJECT_DB.PUBLIC.WORKFLOW_RUNS
        WHERE WORKFLOW_NAME = 'similar_tickets'
        ORDER BY RUN_AT DESC
        LIMIT 10
    """).to_pandas()
    display(runs_df)
else:
    print("WORKFLOW_RUNS absente : impossible d'afficher les journaux de la tâche.")

In [ ]:
# L'historique de la tâche se trouve dans la base PROJECT_DB.
# Une exécution peut être en état SCHEDULED avant de devenir SUCCEEDED.
try:
    session.sql("""
        SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, ERROR_MESSAGE
        FROM TABLE(PROJECT_DB.INFORMATION_SCHEMA.TASK_HISTORY(
            TASK_NAME => 'TASK_SIMILAR_TICKETS',
            RESULT_LIMIT => 10
        ))
        ORDER BY SCHEDULED_TIME DESC
    """).show()
except Exception as exc:
    print("Historique indisponible (droits, tâche absente ou contexte Snowflake) :", str(exc)[:300])

---
## 7. Deux situations complémentaires : demander une précision

Ces cas vont **au-delà du minimum**. Ils montrent la troisième branche de l'orchestrateur, `clarify`, et la manière dont le système répond à un dossier incomplet ou ambigu.

- **S5** : plusieurs lignes de prix candidates → demander le type de conteneur / la référence précise.
- **S7** : route POL/POD absente → demander les champs nécessaires avant d'interroger les données.

In [ ]:
RUN_EXTRA_CASES = True
if RUN_EXTRA_CASES:
    r5 = run_demo("S5_plusieurs_lignes_candidates", show_details=False)
    r7 = run_demo("S7_champs_manquants", show_details=False)
else:
    print("Scénarios S5 et S7 disponibles, non exécutés (RUN_EXTRA_CASES=False).")

### Autre test facultatif : S6 hors périmètre

Le ticket concerne un type de demande exclu du périmètre du prototype. Il doit être transmis à l'équipe appropriée, sans consultation inutile des bases de prix.

In [ ]:
RUN_OUT_OF_SCOPE = True
if RUN_OUT_OF_SCOPE:
    r6 = run_demo("S6_hors_perimetre", show_details=False)
else:
    print("S6 prêt mais non exécuté.")

---
## 8. Tableau de synthèse et limites

In [ ]:
summary = []
for sid, result in RESULTS.items():
    expected = BY_ID[sid]["expected"]
    summary.append({
        "Scénario": sid,
        "Décision observée": result.get("decision"),
        "Décision attendue": expected.get("decision"),
        "Règle observée": result.get("decision_rule"),
        "Règle attendue": expected.get("rule"),
        "kb_gap": result.get("kb_gap", False),
        "Décision OK": result.get("decision") == expected.get("decision"),
        "Règle OK": result.get("decision_rule") == expected.get("rule"),
        "Lacune OK": result.get("kb_gap", False) == expected.get("kb_gap"),
        "Étapes tracées": len(result.get("trace") or []),
    })
summary_df = pd.DataFrame(summary)
display(summary_df)
if not summary_df.empty:
    checks = ["Décision OK", "Règle OK", "Lacune OK"]
    print("Scénarios entièrement conformes :", int(summary_df[checks].all(axis=1).sum()),
          "/", len(summary_df))
    if not summary_df[checks].all(axis=1).all():
        print("⚠️ Une divergence est un résultat à expliquer, pas à masquer ni à corriger en modifiant les attendus.")